In [1]:
!pip install langchain langchain-community langchain-groq chromadb sentence-transformers pypdf -q

In [2]:
from google.colab import files

uploaded = files.upload()

pdf_name = list(uploaded.keys())[0]

print("Uploaded:", pdf_name)

Saving Attention.pdf to Attention (2).pdf
Uploaded: Attention (2).pdf


In [3]:
!pip install langchain-text-splitters -q

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(pdf_name)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 93


In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Create the embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create the vector database from the document chunks
db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("✅ Vector database ready!")

/tmp/ipykernel_6223/3796508080.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Vector database ready!


In [11]:
!pip uninstall -y langchain langchain-community langchain-core
!pip install langchain==0.2.16 langchain-community==0.2.16 langchain-groq chromadb sentence-transformers pypdf -q

Found existing installation: langchain 1.3.14
Uninstalling langchain-1.3.14:
  Successfully uninstalled langchain-1.3.14
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
Found existing installation: langchain-core 1.4.9
Uninstalling langchain-core-1.4.9:
  Successfully uninstalled langchain-core-1.4.9
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [6]:
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
from google.colab import userdata

# Load your Groq API key from Colab Secrets
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

# Create the LLM
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model="llama-3.1-8b-instant",
    temperature=0
)

# Create the RetrievalQA chain
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=db.as_retriever()
)

# Ask a question
query = "What is this document about?"

# Get the answer
response = qa.invoke({"query": query})

print("Question:", query)
print("\nAnswer:")
print(response["result"])

Question: What is this document about?

Answer:
This document appears to be a research paper titled "Attention Is All You Need" by Ashish Vaswani et al. from Google Brain and Google Research. The paper discusses a new approach to natural language processing (NLP) called the Transformer model, which uses self-attention mechanisms to process input sequences.

The document includes a section called "Attention Visualizations" which shows an example input sequence, but it does not provide a clear summary of the paper's main content. However, based on the title and the context, it seems that the paper introduces a new architecture for NLP tasks, such as language translation and text summarization, that relies solely on self-attention mechanisms, hence the title "Attention Is All You Need".

The paper was published in 2017 and has been a significant contribution to the field of NLP, leading to the development of many state-of-the-art models for various NLP tasks.
